# 03 | All-20 PLSA topic-count study

This study asks how the number of PLSA topics affects retrieval across all 20 categories of 20 Newsgroups. We compare five predeclared topic counts while keeping the data split, vocabulary, queries, and evaluation procedure fixed.

The corpus is divided into training (60%), validation (20%), and test (20%) sets. Training data fit the models; validation MAP@10 selects a topic count separately for each PLSA scoring method; the test set reports the predeclared topic-count curve and the TF-IDF/LSI reference results. Category labels are used to stratify and evaluate retrieval, never as PLSA training targets.


## 1. Settings and imports

The notebook uses the shared preprocessing archive and source modules. Running it replaces the previous files in `results/all20_k_study/`.


In [ ]:
import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

REPOSITORY_ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.evaluation import evaluate_ranked_retrieval
from src.lsi import fit_lsi
from src.plsa_model import train_plsa_restarts
from src.retrieval import cosine_scores, plsa_query_log_likelihood
from src.text_processing import load_nlp, preprocess_texts
from src.tfidf import fit_tfidf

SEED = 42
TRAIN_FRACTION = 0.60
VALIDATION_FRACTION = 0.20
K_GRID = [2, 5, 8, 10, 20]
N_RESTARTS = 3
MAX_ITER = 80
TOLERANCE = 1e-4
MAX_FEATURES = 3000
MIN_DF = 3
MAX_DF = 0.90
RETRIEVAL_K = 10
FOLD_IN_ITERATIONS = 100
DATA_PATH = REPOSITORY_ROOT / 'data' / '20news_preprocessed.npz'
OUTPUT_DIR = REPOSITORY_ROOT / 'results' / 'all20_k_study'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repository: {REPOSITORY_ROOT.resolve()}')
print(f'Results folder: {OUTPUT_DIR.resolve()}')
print(f'PLSA topic counts: {K_GRID}; restarts per count: {N_RESTARTS}')


## 2. Load the all-20 corpus and fixed queries

The corpus was downloaded and preprocessed by `src/preprocessing.ipynb`. Here we load its processed text and original category labels. One manually written query represents each category; the category name defines which held-out documents are relevant to that query.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing {DATA_PATH}. Run src/preprocessing.ipynb first.')
with np.load(DATA_PATH, allow_pickle=False) as corpus:
    documents = corpus['documents'].tolist()
    labels = corpus['labels'].astype(np.int64, copy=False)
    target_names = corpus['category_names'].tolist()

category_queries = {
    'alt.atheism': 'atheism atheist religion belief',
    'comp.graphics': 'computer graphics rendering image pixels',
    'comp.os.ms-windows.misc': 'windows operating system microsoft files',
    'comp.sys.ibm.pc.hardware': 'IBM PC hardware disk memory computer',
    'comp.sys.mac.hardware': 'Apple Macintosh Mac hardware computer',
    'comp.windows.x': 'X Window System display server windows',
    'misc.forsale': 'for sale price shipping offer',
    'rec.autos': 'car automobile engine dealer',
    'rec.motorcycles': 'motorcycle bike riding helmet',
    'rec.sport.baseball': 'baseball players team season game',
    'rec.sport.hockey': 'hockey team NHL puck season',
    'sci.crypt': 'cryptography encryption key security',
    'sci.electronics': 'electronics circuit voltage resistor',
    'sci.med': 'medical doctor disease treatment health',
    'sci.space': 'space NASA spacecraft orbit planet',
    'soc.religion.christian': 'Christian church Bible Jesus faith',
    'talk.politics.guns': 'gun control firearms second amendment',
    'talk.politics.mideast': 'Middle East Israel Arab Palestinian',
    'talk.politics.misc': 'government election political policy',
    'talk.religion.misc': 'religion religious belief faith',
}
if set(category_queries) != set(target_names):
    raise ValueError('Queries must cover every source category exactly once.')

category_counts = pd.DataFrame({
    'category': target_names,
    'documents': [int(np.count_nonzero(labels == i)) for i in range(len(target_names))],
})
print(f'Documents: {len(documents):,}; categories: {len(target_names)}')
display(category_counts)


## 3. Make the fixed train/validation/test split and count features

We first reserve 40% of documents, then split that portion evenly into validation and test sets. Stratification preserves category representation in all three sets. The count vocabulary is learned from training text only; validation and test documents are transformed with that vocabulary. Empty processed documents are excluded from retrieval candidates for every method.


In [ ]:
docs_train, docs_temp, y_train, y_temp = train_test_split(
    documents, labels, test_size=1 - TRAIN_FRACTION,
    random_state=SEED, stratify=labels,
)
relative_test_size = (1 - TRAIN_FRACTION - VALIDATION_FRACTION) / (1 - TRAIN_FRACTION)
docs_validation, docs_test, y_validation, y_test = train_test_split(
    docs_temp, y_temp, test_size=relative_test_size,
    random_state=SEED, stratify=y_temp,
)

from sklearn.feature_extraction.text import CountVectorizer
count_vectorizer = CountVectorizer(
    lowercase=False, tokenizer=str.split, token_pattern=None,
    max_features=MAX_FEATURES, min_df=MIN_DF, max_df=MAX_DF,
    dtype=np.float64,
)
X_train = count_vectorizer.fit_transform(docs_train)
X_validation = count_vectorizer.transform(docs_validation)
X_test = count_vectorizer.transform(docs_test)
vocabulary = count_vectorizer.get_feature_names_out()

nonempty_validation = np.asarray([bool(text.strip()) for text in docs_validation])
nonempty_test = np.asarray([bool(text.strip()) for text in docs_test])
split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'documents': [len(docs_train), len(docs_validation), len(docs_test)],
    'empty_documents': [
        sum(not text.strip() for text in docs_train),
        int((~nonempty_validation).sum()), int((~nonempty_test).sum()),
    ],
})
display(split_summary)
print('Count matrix shapes:', X_train.shape, X_validation.shape, X_test.shape)


## 4. Fit the TF-IDF and LSI reference models

TF-IDF and LSI are each fitted once using training documents only. They provide fixed retrieval reference lines for the PLSA topic-count comparison; their representations do not depend on PLSA's K.


In [ ]:
tfidf_vectorizer, (X_train_tfidf, X_validation_tfidf, X_test_tfidf) = fit_tfidf(
    docs_train, docs_validation, docs_test,
    max_features=MAX_FEATURES, min_df=MIN_DF, max_df=MAX_DF,
)
lsi_model, (lsi_train, lsi_validation, lsi_test) = fit_lsi(
    X_train_tfidf, X_validation_tfidf, X_test_tfidf,
    n_components=50, seed=SEED,
)
print('TF-IDF shapes:', X_train_tfidf.shape, X_validation_tfidf.shape, X_test_tfidf.shape)
print('LSI shapes:', lsi_train.shape, lsi_validation.shape, lsi_test.shape)


## 5. Define the query evaluation procedure

Each category query ranks the non-empty documents in a held-out split. PLSA is evaluated with two scoring rules: cosine similarity between folded-in topic mixtures, and query log-likelihood under each document's PLSA word distribution. TF-IDF and LSI use cosine similarity. We report Precision@10, Recall@10, AP@10, and NDCG@10 for each query; the mean AP across the 20 queries is MAP@10.


In [ ]:
query_nlp = load_nlp()

def evaluate_split(split_name, candidate_labels, candidate_tfidf,
                   candidate_lsi, eligible_mask, plsa_model=None, candidate_theta=None,
                   k_topics=None):
    eligible_labels = np.asarray(candidate_labels)[eligible_mask]
    eligible_tfidf = candidate_tfidf[eligible_mask]
    eligible_lsi = candidate_lsi[eligible_mask]
    eligible_theta = candidate_theta[eligible_mask] if candidate_theta is not None else None
    rows = []

    for category, query_text in category_queries.items():
        target_label = target_names.index(category)
        processed_query = preprocess_texts([query_text], nlp=query_nlp)[0]
        query_counts = count_vectorizer.transform([processed_query]).astype(np.float64)
        query_tfidf = tfidf_vectorizer.transform([processed_query])
        query_lsi = lsi_model.transform(query_tfidf)
        method_scores = {}
        if plsa_model is None:
            method_scores = {
                'TF-IDF/VSM': cosine_scores(query_tfidf, eligible_tfidf),
                'LSI': cosine_scores(query_lsi, eligible_lsi),
            }
        else:
            query_theta = plsa_model.fold_in(query_counts, max_iter=FOLD_IN_ITERATIONS)
            method_scores['PLSA cosine'] = cosine_scores(query_theta, eligible_theta)
            method_scores['PLSA query likelihood'] = plsa_query_log_likelihood(
                query_counts, eligible_theta, plsa_model.phi_, smoothing=1e-12,
            )

        for method, scores in method_scores.items():
            metrics = evaluate_ranked_retrieval(
                scores, eligible_labels, target_label, k=RETRIEVAL_K,
            )
            rows.append({
                'split': split_name, 'K': k_topics, 'query_category': category,
                'query_text': query_text, 'method': method,
                'Precision@10': metrics['Precision@k'],
                'Recall@10': metrics['Recall@k'],
                'AP@10': metrics['AP@k'], 'NDCG@10': metrics['NDCG@k'],
                'relevant_documents': int(np.count_nonzero(eligible_labels == target_label)),
            })
    return rows


## 6. Fit PLSA across the predeclared topic counts

For every K, we fit three independent initializations and retain the model with the largest training log-likelihood. This section fits and records all five PLSA models. After fitting is complete, the next section evaluates them on validation and test documents with each learned topic-word distribution held fixed. Validation MAP@10 selects K separately for cosine and query-likelihood ranking. Test scores across all five K values form the planned sensitivity comparison; they are not used to change the experiment settings.


In [ ]:
model_order_rows = []
models_by_k = {}

for k_topics in K_GRID:
    print(f'\nFitting PLSA with K={k_topics} ({N_RESTARTS} restarts)')
    model, runs = train_plsa_restarts(
        X_train, n_topics=k_topics, n_restarts=N_RESTARTS,
        max_iter=MAX_ITER, tol=TOLERANCE,
        base_seed=SEED + 100 * k_topics,
    )
    models_by_k[k_topics] = model
    model_order_rows.append({
        'K': k_topics, 'selected_iterations': model.n_iter_,
        'best_training_log_likelihood': float(model.log_likelihoods[-1]),
        'restart_count': len(runs),
        'restart_training_log_likelihoods': [float(ll) for _, ll in runs],
    })
model_order = pd.DataFrame(model_order_rows)
display(model_order[['K', 'selected_iterations', 'best_training_log_likelihood', 'restart_count']])


## 7. Select topic counts on validation and evaluate fixed baselines

The selection criterion is validation MAP@10. The PLSA cosine and query-likelihood scoring methods may prefer different K values, so each gets its own validation-selected count. We also evaluate TF-IDF and LSI on validation and test once each, since their fitted representations do not vary with K.


In [ ]:
validation_rows = []
test_plsa_rows = []
for k_topics, model in models_by_k.items():
    print(f'Evaluating held-out documents for K={k_topics}')
    theta_validation = model.fold_in(X_validation, max_iter=FOLD_IN_ITERATIONS)
    theta_test = model.fold_in(X_test, max_iter=FOLD_IN_ITERATIONS)
    validation_rows.extend(evaluate_split(
        'validation', y_validation, X_validation_tfidf,
        lsi_validation, nonempty_validation, plsa_model=model,
        candidate_theta=theta_validation, k_topics=k_topics,
    ))
    test_plsa_rows.extend(evaluate_split(
        'test', y_test, X_test_tfidf, lsi_test, nonempty_test,
        plsa_model=model, candidate_theta=theta_test, k_topics=k_topics,
    ))
validation_by_query = pd.DataFrame(validation_rows)
test_plsa_by_query = pd.DataFrame(test_plsa_rows)

validation_baselines = pd.DataFrame(evaluate_split(
    'validation', y_validation, X_validation_tfidf,
    lsi_validation, nonempty_validation,
))
test_baselines = pd.DataFrame(evaluate_split(
    'test', y_test, X_test_tfidf, lsi_test, nonempty_test,
))

metric_columns = ['Precision@10', 'Recall@10', 'AP@10', 'NDCG@10']
validation_plsa_summary = (
    validation_by_query.groupby(['K', 'method'], as_index=False)[metric_columns].mean()
)
validation_baseline_summary = (
    validation_baselines.groupby('method', as_index=False)[metric_columns].mean()
)
validation_map = validation_plsa_summary[validation_plsa_summary['method'].str.startswith('PLSA')]
best_k_by_method = {
    method: int(group.sort_values(['AP@10', 'Precision@10'], ascending=False).iloc[0]['K'])
    for method, group in validation_map.groupby('method')
}
validation_baseline_summary = validation_baseline_summary.rename(columns={'AP@10': 'MAP@10'})
validation_plsa_summary = validation_plsa_summary.rename(columns={'AP@10': 'MAP@10'})
print('Validation-selected topic count by PLSA scoring method:')
display(pd.DataFrame([
    {'method': method, 'selected_K': k}
    for method, k in best_k_by_method.items()
]))
print('Validation results for PLSA by topic count:')
display(validation_plsa_summary.sort_values(['method', 'K']).round(4))
print('Validation reference results:')
display(validation_baseline_summary.round(4))


## 8. Compare held-out test retrieval results

The test summary reports the PLSA retrieval measures at every predeclared K and the two fixed reference methods once. MAP@10 is the macro-average of the 20 query-level AP@10 scores. Interpret the full PLSA curve as a sensitivity analysis; use validation, rather than the test curve, when naming the selected K for a scoring method.


In [ ]:
test_plsa_summary = (
    test_plsa_by_query.groupby(['K', 'method'], as_index=False)[metric_columns].mean()
    .rename(columns={'AP@10': 'MAP@10'})
)
test_baseline_summary = (
    test_baselines.groupby('method', as_index=False)[metric_columns].mean()
    .rename(columns={'AP@10': 'MAP@10'})
)
test_by_query = pd.concat([test_plsa_by_query, test_baselines], ignore_index=True)
test_summary = pd.concat([test_plsa_summary, test_baseline_summary], ignore_index=True)
selected_test_plsa = test_plsa_summary[
    test_plsa_summary.apply(
        lambda row: int(row['K']) == best_k_by_method[row['method']], axis=1,
    )
].copy()
selected_test_summary = pd.concat([selected_test_plsa, test_baseline_summary], ignore_index=True)

print('Held-out test results at validation-selected K, alongside fixed baselines:')
display(selected_test_summary.round(4))
print('Held-out PLSA results by K:')
display(test_plsa_summary.sort_values(['method', 'K']).round(4))
print('Held-out TF-IDF and LSI reference results:')
display(test_baseline_summary.round(4))


In [ ]:
# Validation MAP@10 by K for the two PLSA scoring methods.
validation_map_plot = validation_plsa_summary.pivot(
    index='K', columns='method', values='MAP@10',
).sort_index()
validation_fig, validation_ax = plt.subplots(figsize=(8, 5))
for method in validation_map_plot.columns:
    validation_ax.plot(validation_map_plot.index, validation_map_plot[method], marker='o', label=method)
for method, selected_k in best_k_by_method.items():
    validation_ax.axvline(selected_k, linestyle='--', alpha=0.35)
validation_ax.set(title='Validation MAP@10 by PLSA topic count', xlabel='Topic count K', ylabel='Macro-average AP@10')
validation_ax.set_xticks(K_GRID)
validation_ax.grid(alpha=0.3)
validation_ax.legend()
validation_fig.tight_layout()
# display(validation_fig)

# Test MAP@10 by K, with fixed TF-IDF and LSI references.
test_map_plot = test_plsa_summary.pivot(index='K', columns='method', values='MAP@10').sort_index()
test_fig, test_ax = plt.subplots(figsize=(8, 5))
for method in test_map_plot.columns:
    test_ax.plot(test_map_plot.index, test_map_plot[method], marker='o', label=method)
for _, row in test_baseline_summary.iterrows():
    test_ax.axhline(row['MAP@10'], linestyle='--', label=f"{row['method']} reference")
test_ax.set(title='Held-out test MAP@10 across the PLSA topic-count study', xlabel='PLSA topic count K', ylabel='Macro-average AP@10')
test_ax.set_xticks(K_GRID)
test_ax.grid(alpha=0.3)
test_ax.legend()
test_fig.tight_layout()
# display(test_fig)


## 9. Inspect query-level results

Macro-averages can hide categories where a method performs unusually well or poorly. This table keeps the held-out test AP@10 values for each query visible alongside K and scoring method.


In [ ]:
test_ap_by_query = test_by_query.pivot_table(
    index='query_category', columns=['K', 'method'], values='AP@10',
)
display(test_ap_by_query.round(4))


## 10. Save outputs

Tables, figures, and run settings are written to `results/all20_k_study/`. Rerunning this notebook replaces the previous outputs for this experiment.


In [ ]:
model_order.to_json(OUTPUT_DIR / 'plsa_model_order.json', orient='records', indent=2)
validation_by_query.to_csv(OUTPUT_DIR / 'validation_metrics_by_query.csv', index=False)
validation_plsa_summary.to_csv(OUTPUT_DIR / 'validation_plsa_summary.csv', index=False)
validation_baseline_summary.to_csv(OUTPUT_DIR / 'validation_baseline_summary.csv', index=False)
test_by_query.to_csv(OUTPUT_DIR / 'test_metrics_by_query.csv', index=False)
test_summary.to_csv(OUTPUT_DIR / 'test_metrics_summary.csv', index=False)
selected_test_summary.to_csv(OUTPUT_DIR / 'test_metrics_at_validation_selected_k.csv', index=False)
test_ap_by_query.to_csv(OUTPUT_DIR / 'test_ap_by_query.csv')
validation_fig.savefig(OUTPUT_DIR / 'validation_map_by_k.png', dpi=160, bbox_inches='tight')
test_fig.savefig(OUTPUT_DIR / 'test_map_by_k.png', dpi=160, bbox_inches='tight')

configuration = {
    'experiment': 'All-20 PLSA topic-count study',
    'seed': SEED,
    'split_fractions': {'train': TRAIN_FRACTION, 'validation': VALIDATION_FRACTION, 'test': 1 - TRAIN_FRACTION - VALIDATION_FRACTION},
    'k_grid': K_GRID,
    'restarts_per_k': N_RESTARTS,
    'max_iter': MAX_ITER,
    'tolerance': TOLERANCE,
    'max_features': MAX_FEATURES,
    'min_df': MIN_DF,
    'max_df': MAX_DF,
    'retrieval_k': RETRIEVAL_K,
    'selection_metric': 'validation MAP@10, independently for each PLSA scoring method',
    'validation_selected_k': best_k_by_method,
    'categories': target_names,
    'query_set': category_queries,
    'documents': len(documents),
}
with (OUTPUT_DIR / 'configuration.json').open('w', encoding='utf-8') as handle:
    json.dump(configuration, handle, indent=2)
print(f'Saved all-20 K-study outputs to {OUTPUT_DIR.resolve()}')
